# QML Lab 2 — Variational classifiers and quantum neural networks
### Qiskit Fall Fest · University of Ottawa

**Question:** how do we train a quantum circuit, and what changes when its outputs are noisy?

This lab accompanies the
[variational classifiers and QNNs lesson](https://quantum.cloud.ibm.com/learning/en/courses/quantum-machine-learning/qvc-qnn)
of IBM's QML course. It is self-contained: you do not need variables or saved results from Lab 1.

**Prerequisites:** Python/NumPy, parameterized circuits, expectation values, train/validation/test
splits. Lab 1 is recommended. The optional hardware section builds on Notebook 04.

**Suggested teaching time:** 110–130 minutes. The core is §§1–4 and §6; §5 adds the QNN
software interface, and §§7–8 are extensions. Local training time varies by machine.

### By the end, you can

1. Separate data parameters, trainable weights, readout, loss, and optimizer.
2. Train a variational quantum classifier (VQC) and inspect validation performance.
3. Derive and check a parameter-shift gradient for a compatible circuit.
4. Use the same model through Qiskit Machine Learning's `EstimatorQNN` interface.
5. Separate finite-shot effects from modeled gate/readout noise.
6. Explain architecture choices, data re-uploading, and barren-plateau concerns.

**Execution policy:** all default work runs locally. The hardware section has an explicit
opt-in flag and a circuit budget. It performs inference only, using weights trained locally.

## Setup — run this first

Use **Python 3.12 or newer**. Qiskit **2.5.2** was the latest stable release on
**29 September 2026**; the lab pins that release and the tested companion packages.
These notebooks use V2 primitives and the current circuit-library functions.

- **Colab:** select a Python runtime, change `INSTALL_PACKAGES` to `True`, and run the next cell.
- **Local Jupyter:** select your workshop environment. You can install the same dependencies
  with `python -m pip install -r notebooks/requirements-qml.txt` from the repository root.
- After installing into an already running kernel, **restart the kernel**, set
  `INSTALL_PACKAGES = False`, and run the notebook from the top.

All default cells run locally. There are no accounts, tokens, dataset downloads, or paid jobs
in the default path. Run cells in order. Exercises ask you to change a working baseline, so
**Run All** works before you start editing.

Version sources: [Qiskit on PyPI](https://pypi.org/project/qiskit/),
[Qiskit Machine Learning](https://pypi.org/project/qiskit-machine-learning/),
[Qiskit Aer](https://pypi.org/project/qiskit-aer/).

In [ ]:
INSTALL_PACKAGES = False  # Set True once in a fresh Colab/runtime, then restart the kernel.
if INSTALL_PACKAGES:
    import subprocess
    import sys
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        "qiskit[visualization]==2.5.2", "qiskit-aer==0.17.2",
        "qiskit-machine-learning==0.9.1", "qiskit-ibm-runtime==0.50.0",
        "scipy==1.18.1", "scikit-learn==1.9.1",
        "matplotlib==3.11.2", "pandas==3.0.6",
    ])
    print("Installation complete. Restart the kernel before continuing.")

In [ ]:
from importlib.metadata import version
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit.primitives import StatevectorSampler, StatevectorEstimator

SEED = 27
np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.figsize": (6, 4), "font.size": 11})
display(pd.DataFrame({
    "package": ["qiskit", "qiskit-aer", "qiskit-machine-learning", "scikit-learn"],
    "installed": [version(p) for p in
                  ["qiskit", "qiskit-aer", "qiskit-machine-learning", "scikit-learn"]],
}))
if version("qiskit") != "2.5.2":
    print("This environment differs from the tested Qiskit 2.5.2 environment.")

## 1. A small image-classification task · 10 min

Each image contains four pixels, flattened in this order:

$$\begin{pmatrix}x_0 & x_1\\x_2 & x_3\end{pmatrix}.$$

Class 0 contains horizontal stripes; class 1 contains vertical stripes. Bright/dark polarity
is randomized independently of class so the model must recognize **relationships between
pixels**, not just the value of one pixel. Additive noise makes the examples imperfect.
Pixel values are clipped to the known physical range $[0,1]$; this is fixed by the data
specification and does not require fitting a scaler on held-out data.

We use 40 training, 20 validation, and 20 test examples. The test split stays unused until
we have chosen a trained model. All datasets are generated locally and seeded.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay


def make_stripes(n_per_class=40, pixel_noise=0.08, seed=SEED):
    rng = np.random.default_rng(seed)
    labels = np.tile([0, 1], n_per_class)
    images = []
    for label in labels:
        a, b = (0.1, 0.9) if rng.random() < 0.5 else (0.9, 0.1)
        image = np.array([[a, a], [b, b]]) if label == 0 else np.array([[a, b], [a, b]])
        images.append(np.clip(image + rng.normal(0, pixel_noise, (2, 2)), 0, 1))
    return np.array(images).reshape(-1, 4), labels

X, y = make_stripes()
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=1 / 3, stratify=y_dev, random_state=SEED
)
fig, axes = plt.subplots(2, 4, figsize=(8, 4))
for ax, pixels, label in zip(axes.flat, X_train[:8], y_train[:8]):
    ax.imshow(pixels.reshape(2, 2), cmap="gray", vmin=0, vmax=1)
    ax.set_title("horizontal" if label == 0 else "vertical")
    ax.axis("off")
plt.tight_layout()
plt.show()
print("Split sizes:", len(X_train), len(X_val), len(X_test))

## 2. Build the trainable model · 15 min

The model has five separate ingredients:

| Component | This notebook | Classical analogue |
|---|---|---|
| Data encoding $U(x)$ | $R_y(\pi x_i)$ on pixel $i$ | Input representation |
| Ansatz $W(\theta)$ | Trainable $R_y$ rotations and a CNOT chain | Layers with weights |
| Readout $O$ | Four-qubit Z parity | Scalar model output |
| Loss | Mean squared error (MSE) | Prediction error |
| Optimizer | COBYLA, using loss evaluations | Parameter updates |

The expectation is
$$f_\theta(x)=\langle0|U(x)^\dagger W(\theta)^\dagger O W(\theta)U(x)|0\rangle.$$
For parity, $O=Z_3Z_2Z_1Z_0$, so $-1\le f\le1$. Map labels to $t=1-2y$:
class 0 targets +1, class 1 targets −1. Predict class 1 when $f<0$.
The odd-parity measurement probability is $p_1=(1-f)/2$; it is a model score, not automatically
a calibrated probability that the real-world label is 1.

We minimize $L(\theta)=\frac{1}{N}\sum_i(f_\theta(x_i)-t_i)^2$.
A classical neural network also combines a parameterized function, loss, and optimizer.
The quantum model obtains its function values from circuit measurements.

**Qiskit convention:** the leftmost character of a Pauli label acts on the highest-index qubit.
For example, `IIZZ` measures the correlation of pixels 0 and 1.

In [ ]:
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import efficient_su2

n_qubits = 4
input_parameters = list(ParameterVector("x", n_qubits))
encoding = QuantumCircuit(n_qubits, name="pixels")
for q, parameter in enumerate(input_parameters):
    encoding.ry(np.pi * parameter, q)

ansatz = efficient_su2(n_qubits, su2_gates=["ry"], entanglement="linear", reps=1,
                       parameter_prefix="theta")
weight_parameters = list(ansatz.parameters)
model_circuit = encoding.compose(ansatz)
observable = SparsePauliOp("ZZZZ")
estimator = StatevectorEstimator(seed=SEED)
parameter_order = list(model_circuit.parameters)


def parameter_rows(points, weights):
    points = np.atleast_2d(np.asarray(points, dtype=float))
    weights = np.asarray(weights, dtype=float)
    if points.shape[1] != len(input_parameters) or weights.shape != (len(weight_parameters),):
        raise ValueError("Input or weight dimension mismatch.")
    rows = []
    for point in points:
        binding = {**dict(zip(input_parameters, point)), **dict(zip(weight_parameters, weights))}
        rows.append([binding[p] for p in parameter_order])
    return np.asarray(rows)


def expectations(points, weights):
    # One V2 primitive input contains a batch of parameter bindings.
    pub = (model_circuit, observable, parameter_rows(points, weights))
    return np.asarray(estimator.run([pub], precision=0.0).result()[0].data.evs)


def mse_loss(weights, points, labels):
    return float(np.mean((expectations(points, weights) - (1 - 2 * labels)) ** 2))

initial_weights = np.random.default_rng(SEED).uniform(-np.pi, np.pi, len(weight_parameters))
initial_values = expectations(X_train, initial_weights)
assert initial_values.shape == (len(X_train),)
assert np.all(np.abs(initial_values) <= 1 + 1e-12)
print(f"{n_qubits} input values; {len(weight_parameters)} trainable weights")
print("Initial training MSE:", mse_loss(initial_weights, X_train, y_train))
display(model_circuit.draw("mpl", fold=90))

## 3. Train with a classical optimizer · 25 min

COBYLA uses objective values rather than analytic derivatives. Each objective evaluation
runs the model on the training examples. Here those are exact local expectations. On a QPU,
each evaluation would require measurement shots, and the resulting loss would be noisy.

We try two **predeclared initializations** and choose between them using validation MSE.
We do not select weights from test data. The optimizer receives only training loss.
A fixed evaluation budget makes the workshop predictable; hitting it does not mean the model
has converged. Record the optimizer's message as well as its score.

**During training:** predict whether adding more circuit layers necessarily improves the result.

In [ ]:
from scipy.optimize import minimize
from time import perf_counter

MAX_EVALUATIONS = 140  # Raise for a follow-up study, using validation to judge the benefit.
training_runs = []
start = perf_counter()
for initialization_seed in [SEED, SEED + 11]:
    theta0 = np.random.default_rng(initialization_seed).uniform(-np.pi, np.pi, len(weight_parameters))
    trace = []

    def objective(theta):
        loss = mse_loss(theta, X_train, y_train)
        trace.append(loss)
        return loss

    fit = minimize(objective, theta0, method="COBYLA",
                   options={"maxiter": MAX_EVALUATIONS, "rhobeg": 0.7})
    run = {"seed": initialization_seed, "fit": fit, "trace": trace,
           "validation_mse": mse_loss(fit.x, X_val, y_val)}
    training_runs.append(run)
    print(f"Seed {initialization_seed}: training MSE={fit.fun:.4f}, "
          f"validation MSE={run['validation_mse']:.4f}, evaluations={fit.nfev}")
    print("  Optimizer status:", fit.message)
selected_run = min(training_runs, key=lambda run: run["validation_mse"])
trained_weights = selected_run["fit"].x.copy()
print(f"Selected seed {selected_run['seed']}; training elapsed {perf_counter() - start:.1f} s")

fig, ax = plt.subplots()
for run in training_runs:
    ax.plot(run["trace"], label=f"seed {run['seed']}", alpha=0.8)
ax.set(xlabel="Objective evaluation (not gradient step)", ylabel="Training MSE",
       title="Training trajectory: trial points may increase the loss")
ax.legend()
plt.show()

In [ ]:
# Small, explicit classical comparisons: same inputs, training split, and validation set.
classical_runs = []
for C in [0.1, 1.0, 10.0]:
    classical = SVC(C=C, kernel="rbf", gamma="scale").fit(X_train, y_train)
    classical_runs.append({"name": f"RBF SVM C={C}", "model": classical,
                           "validation_accuracy": classical.score(X_val, y_val)})
for width in [4, 8]:
    classical = MLPClassifier(hidden_layer_sizes=(width,), solver="lbfgs", alpha=0.01,
                              max_iter=500, random_state=SEED).fit(X_train, y_train)
    classical_runs.append({"name": f"Classical NN ({width} hidden units)", "model": classical,
                           "validation_accuracy": classical.score(X_val, y_val)})
selected_classical = {
    family: max([run for run in classical_runs if run["name"].startswith(family)],
                key=lambda run: run["validation_accuracy"])
    for family in ["RBF", "Classical NN"]
}
display(pd.DataFrame([{k: v for k, v in row.items() if k != "model"} for row in classical_runs]))

# Settings are frozen. Open the test split once for the selected models.
test_expectations = expectations(X_test, trained_weights)
test_predictions = (test_expectations < 0).astype(int)
report = [{"model": "Variational quantum classifier",
           "validation_accuracy": accuracy_score(y_val, expectations(X_val, trained_weights) < 0),
           "test_accuracy": accuracy_score(y_test, test_predictions)}]
for run in selected_classical.values():
    report.append({"model": run["name"], "validation_accuracy": run["validation_accuracy"],
                   "test_accuracy": run["model"].score(X_test, y_test)})
display(pd.DataFrame(report))
ConfusionMatrixDisplay.from_predictions(y_test, test_predictions,
                                       display_labels=["horizontal", "vertical"])
plt.title("Held-out VQC predictions")
plt.show()

### Exercise 1 — Explain the training loop

Point to the code that performs each step: encode inputs, bind weights, estimate a readout,
compute a loss, propose new weights, and choose an initialization.

Then describe one situation in which low training loss and poor validation accuracy occur.

<details><summary>Checkpoint</summary>

`parameter_rows` binds features and weights; `expectations` invokes the estimator;
`mse_loss` compares outputs to targets; `minimize` proposes weights; `selected_run` uses
validation MSE. Overfitting, an unrepresentative split, or distribution shift can separate
training and validation performance. More layers can increase both fitting capacity and
optimization/measurement cost.

</details>

## 4. Where do gradients come from? · 20 min

Each weight in our circuit occurs in exactly **one** $R_y(\theta_j)$ gate. For this form,

$$\partial_j f(\theta)=\tfrac12[f(\theta+\tfrac\pi2 e_j)-f(\theta-\tfrac\pi2 e_j)].$$

This is the **parameter-shift rule** for an expectation, not a finite-difference approximation.
For our MSE, apply the chain rule:

$$\partial_jL=\frac{2}{N}\sum_i(f_i-t_i)\,\partial_jf_i.$$

Do **not** substitute the nonlinear loss into the two-shift expectation formula.
If one weight is reused in several gates, or a gate has another generator spectrum, the simple
formula above must be adapted. The implementation here is deliberately limited to this circuit.

For $P$ parameters and $N$ examples, a full shift gradient needs $2PN$ expectation evaluations,
plus $N$ for the unshifted predictions. Batching reduces API calls; it does not remove the
underlying circuits or shot cost.

In [ ]:
def parameter_shift_jacobian(points, weights):
    jac = np.empty((len(points), len(weights)))
    for j in range(len(weights)):
        shift = np.zeros_like(weights)
        shift[j] = np.pi / 2
        jac[:, j] = (expectations(points, weights + shift) -
                     expectations(points, weights - shift)) / 2
    return jac

# Use a small training batch for a quick, transparent verification.
X_batch, y_batch = X_train[:6], y_train[:6]
# Checking away from an optimum avoids a trivially tiny gradient comparison.
weights_to_check = initial_weights.copy()
J = parameter_shift_jacobian(X_batch, weights_to_check)
residual = expectations(X_batch, weights_to_check) - (1 - 2 * y_batch)
shift_loss_gradient = 2 * J.T @ residual / len(X_batch)

epsilon = 1e-5
finite_difference = np.empty(len(weights_to_check))
for j in range(len(weights_to_check)):
    step = np.zeros_like(weights_to_check)
    step[j] = epsilon
    finite_difference[j] = (mse_loss(weights_to_check + step, X_batch, y_batch) -
                             mse_loss(weights_to_check - step, X_batch, y_batch)) / (2 * epsilon)
np.testing.assert_allclose(shift_loss_gradient, finite_difference, atol=1e-7, rtol=1e-5)
print("Gradient checks passed. Maximum absolute difference:",
      np.max(np.abs(shift_loss_gradient - finite_difference)))
print("Expectation evaluations for one full training gradient:",
      (2 * len(weight_parameters) + 1) * len(X_train))

# A small gradient step on this same batch should be inspected, not assumed to help.
learning_rate = 0.05
updated_weights = weights_to_check - learning_rate * shift_loss_gradient
print("Batch MSE before/after one gradient step:",
      mse_loss(weights_to_check, X_batch, y_batch), mse_loss(updated_weights, X_batch, y_batch))

### Exercise 2 — Differentiate the right quantity

Write the gradient of MSE on paper, then identify its two factors in the code. Increase
`learning_rate` to see whether the loss still decreases. Why is parameter shift still useful
if COBYLA did not use it?

<details><summary>Checkpoint</summary>

The factors are the residual $f-t$ and the expectation Jacobian. Large steps can overshoot;
the gradient is local information. Gradient-based optimizers and hybrid classical–quantum
models need derivatives. Shot noise can make those derivatives uncertain and expensive.

</details>

## 5. The same circuit as a QNN · 15 min

A **quantum neural network** is a trainable quantum model presented through an interface
that accepts inputs and weights and returns outputs (and optionally derivatives).
It need not reproduce a classical neuron's activation function.

`EstimatorQNN` wraps our circuit, observable, and V2 estimator. We pass explicit input and
weight parameter lists. Its `forward` should match our manual function; `backward` exposes
the derivatives needed by a classical training loop. `input_gradients=True` also permits
training a classical layer that produces the quantum circuit's inputs.

We choose exact precision for this consistency check. Input derivatives must include the
$\pi$ factor in $R_y(\pi x)$, which the gradient machinery handles. A deep-learning connector
such as `TorchConnector` is an optional integration layer; PyTorch is not needed for this lab.

In [ ]:
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.gradients import ParamShiftEstimatorGradient

qnn = EstimatorQNN(
    circuit=model_circuit,
    estimator=estimator,
    observables=observable,
    input_params=input_parameters,
    weight_params=weight_parameters,
    gradient=ParamShiftEstimatorGradient(estimator),
    input_gradients=True,
    default_precision=0.0,
)
qnn_forward = qnn.forward(X_batch, weights_to_check)
input_jacobian, weight_jacobian = qnn.backward(X_batch, weights_to_check)
np.testing.assert_allclose(qnn_forward[:, 0], expectations(X_batch, weights_to_check), atol=1e-10)
np.testing.assert_allclose(weight_jacobian[:, 0, :], J, atol=1e-9)
assert input_jacobian.shape == (len(X_batch), 1, len(input_parameters))
print("Forward shape (examples, outputs):", qnn_forward.shape)
print("Input gradient shape (examples, outputs, inputs):", input_jacobian.shape)
print("Weight gradient shape (examples, outputs, weights):", weight_jacobian.shape)
print("Manual and QNN outputs/weight gradients agree.")

## 6. Shots and noise are different effects · 20 min

Freeze the trained weights. First sample the ideal circuit, then add a specified Aer noise
model. This comparison measures the effect on **inference**; it does not simulate training
with a noisy QPU in the loop.

For a Z-parity measurement, each bit string contributes +1 for an even number of ones and −1
for an odd number. Average these signs to estimate $f$. Its standard error is approximately
$\sqrt{(1-\hat f^2)/S}$. Increasing shots reduces sampling uncertainty but does not remove
systematic bias caused by gate or readout errors.

Our illustrative noise model has 0.2% single-qubit depolarizing error, 1% two-qubit error,
and 2% independent readout bit flips. These are chosen teaching parameters, **not a calibration
of an IBM device**. Circuits are compiled into the same basis for both comparisons.

In [ ]:
from qiskit_aer.primitives import SamplerV2 as AerSamplerV2
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError

INFERENCE_SHOTS = 2048
noise_model = NoiseModel()
noise_model.add_all_qubit_quantum_error(depolarizing_error(0.002, 1), ["ry"])
noise_model.add_all_qubit_quantum_error(depolarizing_error(0.01, 2), ["cx"])
noise_model.add_all_qubit_readout_error(ReadoutError([[0.98, 0.02], [0.02, 0.98]]))

measured_circuits = []
for values in parameter_rows(X_test, trained_weights):
    bound = model_circuit.assign_parameters(dict(zip(parameter_order, values)))
    bound.measure_all()
    measured_circuits.append(bound)
compiled_circuits = transpile(measured_circuits, basis_gates=["ry", "cx"],
                               optimization_level=0, seed_transpiler=SEED)
ideal_sampler = AerSamplerV2(default_shots=INFERENCE_SHOTS, seed=SEED,
                             options={"backend_options": {"method": "statevector"}})
noisy_sampler = AerSamplerV2(default_shots=INFERENCE_SHOTS, seed=SEED,
                             options={"backend_options": {"noise_model": noise_model,
                                                          "method": "density_matrix"}})


def parity_from_counts(counts):
    total = sum(counts.values())
    return sum(((-1) ** bitstring.replace(" ", "").count("1")) * count
               for bitstring, count in counts.items()) / total


def sample_parities(primitive, circuits):
    return np.array([parity_from_counts(pub.data.meas.get_counts())
                     for pub in primitive.run(circuits).result()])

ideal_sampled = sample_parities(ideal_sampler, compiled_circuits)
noisy_sampled = sample_parities(noisy_sampler, compiled_circuits)
for name, values in [("Exact", test_expectations), ("Ideal shots", ideal_sampled),
                      ("Noisy shots", noisy_sampled)]:
    print(f"{name:12}: accuracy={accuracy_score(y_test, values < 0):.3f}; "
          f"mean |shift from exact|={np.mean(np.abs(values - test_expectations)):.4f}")

fig, ax = plt.subplots(figsize=(9, 4))
indices = np.arange(len(X_test))
ax.plot(indices, test_expectations, "k.", markersize=8, label="exact expectation")
for offset, values, label in [(-0.12, ideal_sampled, "ideal sampling"),
                               (0.12, noisy_sampled, "sampling with modeled noise")]:
    standard_error = np.sqrt(np.maximum(0, 1 - values ** 2) / INFERENCE_SHOTS)
    ax.errorbar(indices + offset, values, yerr=2 * standard_error,
                fmt="o", markersize=3, capsize=2, label=label)
ax.axhline(0, color="gray", linestyle="--", label="decision threshold")
ax.set(xlabel="Held-out example", ylabel="Z-parity expectation",
       title="Frozen model: approximate ±2 sampling standard errors")
ax.legend(fontsize=9)
plt.show()

### Exercise 3 — Does more data collection fix noise?

Increase `INFERENCE_SHOTS` and rerun §6. Predict what will happen to error bars, the systematic
shift from the exact expectation, and classifications near zero. Then change the readout error.
Keep the trained model fixed during this comparison.

<details><summary>Expected interpretation</summary>

Sampling error typically shrinks as $1/\sqrt{S}$. A systematic shift remains unless the noise
or its treatment changes. Accuracy may stay the same even when all expectation values move,
if their signs remain unchanged. Near-zero predictions can change under small perturbations.
For independent symmetric readout flips with probability $e$, an $n$-qubit parity expectation
is multiplied by $(1-2e)^n$; this explains why a global readout can be fragile.

</details>

## 7. Extension — Architecture, re-uploading, and trainability · 15–25 min

### Use the structure of the data

Horizontal stripes correlate pixels 0 and 1; vertical stripes correlate pixels 0 and 2.
The local observable $(Z_0Z_1-Z_0Z_2)/2$ captures that difference. With our label convention,
positive values suggest horizontal stripes. This is a hand-designed representation using
knowledge of the generator. A fair comparison includes its **classical** analogue too.

### Re-uploading

A second encoding layer places data into the circuit again between trainable layers.
It can change the family of functions the circuit can represent, at the cost of more gates
and optimization work. The trainable parameters in each layer below are independent.

### Barren plateaus

Some circuit/loss/initialization families have gradient variance that decreases very rapidly
with system size. Small gradients can become difficult to resolve with finite shots. This is
not the same as reaching a good minimum. Architecture, depth, locality of the cost, noise,
and initialization matter. A four-qubit demonstration cannot establish asymptotic scaling.

In [ ]:
# Inspect architecture ideas on validation data, without retuning the already reported test model.
local_observable = SparsePauliOp.from_list([("IIZZ", 0.5), ("IZIZ", -0.5)])
local_quantum_scores = []
for point in X_val:
    state = Statevector.from_instruction(encoding.assign_parameters(dict(zip(input_parameters, point))))
    local_quantum_scores.append(float(np.real(state.expectation_value(local_observable))))
# A simple classical structural feature: same-row similarity minus same-column similarity.
classical_structural_scores = ((X_val[:, 0] - X_val[:, 2]) ** 2 -
                               (X_val[:, 0] - X_val[:, 1]) ** 2)
print("Engineered quantum feature, validation accuracy:",
      accuracy_score(y_val, np.array(local_quantum_scores) < 0))
print("Engineered classical feature, validation accuracy:",
      accuracy_score(y_val, classical_structural_scores < 0))

reuploaded = QuantumCircuit(n_qubits)
for layer in range(2):
    reuploaded.compose(encoding, inplace=True)
    reuploaded.compose(efficient_su2(n_qubits, su2_gates=["ry"], entanglement="linear",
                                     reps=1, parameter_prefix=f"w{layer}"), inplace=True)
new_weight_parameters = [p for p in reuploaded.parameters if p not in input_parameters]
print("Original / re-uploaded weights:", len(weight_parameters), len(new_weight_parameters))
resource_rows = []
for name, circuit in [("one encoding", model_circuit), ("two encodings", reuploaded)]:
    compiled = transpile(circuit, basis_gates=["ry", "cx"], optimization_level=0)
    resource_rows.append({"circuit": name, "depth": compiled.depth(),
                          "CX_count": compiled.count_ops().get("cx", 0),
                          "weight_count": len([p for p in circuit.parameters if p not in input_parameters])})
display(pd.DataFrame(resource_rows))

### Exercise 4 — Design the next study

Choose **one** extension:

1. Train the re-uploaded circuit. Rebuild parameter binding and a fresh `EstimatorQNN` for its
   new weight list. Keep a fixed objective-evaluation budget and report extra per-circuit cost.
2. Replace global parity with a chosen local readout. Adjust probability/label interpretation
   if the observable's range changes.
3. Compare gradient distributions over random initializations and several qubit/depth settings
   using a new synthetic task. Report variance and shot precision; do not infer a barren plateau
   from one small gradient or one training failure.

Use fresh validation/test splits for a new study after inspecting this notebook's test results.
State the hypothesis and resource budget before running it.

## 8. Optional IBM hardware inference · demonstration only

This section is **disabled by default**. It sends four circuits with 512 shots each only when
you set `RUN_HARDWARE = True`. It does not run an optimizer on hardware.

Before enabling it, configure an authorized IBM Quantum account in your own environment
(see Notebook 04). Never paste an API key into a shared notebook. Check your access and remaining
usage budget, and choose `BACKEND_NAME` explicitly. Device availability and queue times vary.

The preset pass manager handles the device's instruction set and connectivity. For a Sampler,
measurement instructions retain the logical-to-classical mapping. An Estimator workflow instead
needs the observable mapped to the compiled layout, for example `observable.apply_layout(isa.layout)`.

Reference: [IBM primitives examples](https://quantum.cloud.ibm.com/docs/en/guides/primitives-examples).

In [ ]:
RUN_HARDWARE = False
BACKEND_NAME = ""  # Set a backend you are authorized to use; keep credentials outside this notebook.
HARDWARE_SHOTS = 512
MAX_HARDWARE_CIRCUITS = 4

if RUN_HARDWARE:
    if not BACKEND_NAME:
        raise ValueError("Set BACKEND_NAME after checking your account access and budget.")
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as RuntimeSamplerV2
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

    service = QiskitRuntimeService()  # Uses credentials already saved in your environment.
    backend = service.backend(BACKEND_NAME)
    pass_manager = generate_preset_pass_manager(backend=backend, optimization_level=1,
                                                seed_transpiler=SEED)
    isa_circuits = pass_manager.run(measured_circuits[:MAX_HARDWARE_CIRCUITS])
    print(f"Submitting {len(isa_circuits)} circuits × {HARDWARE_SHOTS} shots to {backend.name}.")
    hardware_sampler = RuntimeSamplerV2(mode=backend)
    hardware_job = hardware_sampler.run(isa_circuits, shots=HARDWARE_SHOTS)
    print("Job ID:", hardware_job.job_id())
    # Save the job ID so results can be retrieved later if the queue is long.
else:
    print("Hardware execution disabled. All completed results above are local.")

In [ ]:
# Retrieve later by job ID in a fresh session with service.job("your-job-id").
FETCH_HARDWARE_RESULTS = False
if RUN_HARDWARE and FETCH_HARDWARE_RESULTS:
    hardware_values = np.array([parity_from_counts(pub.data.meas.get_counts())
                                for pub in hardware_job.result()])
    display(pd.DataFrame({"exact_local": test_expectations[:len(hardware_values)],
                          "hardware": hardware_values,
                          "true_label": y_test[:len(hardware_values)]}))
else:
    print("Hardware result retrieval skipped.")

## 9. Exit ticket and project prompt

Explain these distinctions:

- A feature parameter versus a trainable weight.
- A circuit expectation versus a class probability versus a hard prediction.
- Parameter-shift differentiation versus finite differences.
- A shot-limited estimate versus a systematically noisy expectation.
- A model that fits this dataset versus evidence of practical quantum advantage.

### Hackathon prompt — Build and defend a small QML model

Choose a modest classical dataset, define a representation, and implement either a quantum
kernel or a variational classifier. Your submission should include:

1. A working notebook with seeds, package versions, and a diagram of the circuit.
2. A training/validation/test protocol and competitive classical baseline.
3. One controlled ablation: encoding, readout, depth, or noise.
4. A table of qubits, weights, compiled two-qubit gates, circuits, shots, and measured runtime.
5. A short account of when the model fails and what the results do **not** establish.

Do not select a project because its title contains “quantum.” Identify a measurable question
and explain why a quantum representation is worth investigating.

### References

- [IBM course: variational classifiers and QNNs](https://quantum.cloud.ibm.com/learning/en/courses/quantum-machine-learning/qvc-qnn).
- [EstimatorQNN API](https://qiskit-community.github.io/qiskit-machine-learning/stubs/qiskit_machine_learning.neural_networks.EstimatorQNN.html).
- [StatevectorEstimator API](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.primitives.StatevectorEstimator).
- [Qiskit Aer noise models](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.noise.NoiseModel.html).

These workshop exercises are independent learning material; they are not IBM's course assessment.